# ISLR Chapter 6 - Question 9 (College)
Each sub-question is answered below. Every part restates the original prompt before the corresponding code cell.

### Part (a)
**Original prompt:** Split the data set into a training set and a test set.

In [6]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

RANDOM_STATE = 18
data_path = Path('College.csv')
if not data_path.exists():
    raise FileNotFoundError('College.csv not found in this directory.')

college = pd.read_csv(data_path).rename(columns={'Unnamed: 0': 'College'}).set_index('College')
college = pd.get_dummies(college, columns=['Private'], drop_first=True)

X = college.drop(columns='Apps')
y = college['Apps']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE
)

print(f'Training samples: {X_train.shape[0]} | Test samples: {X_test.shape[0]}')
print(f'Number of predictors: {X_train.shape[1]}')
print(f'Random state used for the split: {RANDOM_STATE}')


Training samples: 543 | Test samples: 234
Number of predictors: 17
Random state used for the split: 18


### Part (b)
**Original prompt:** Fit a linear model using least squares on the training set, and report the test error obtained.

In [7]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

ols_model = Pipeline([
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])
ols_model.fit(X_train, y_train)
ols_preds = ols_model.predict(X_test)
ols_mse = mean_squared_error(y_test, ols_preds)
print(f'Test MSE: {ols_mse:,.2f}')
print(f'Test RMSE: {ols_mse ** 0.5:,.2f}')


Test MSE: 1,053,473.23
Test RMSE: 1,026.39


### Part (c)
**Original prompt:** Fit a ridge regression model on the training set, with lambda chosen by cross-validation. Report the test error obtained.

In [8]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error

ridge_alphas = np.logspace(-3, 6, 400)
ridge_model = Pipeline([
    ('center', StandardScaler(with_std=False)),
    ('ridge', RidgeCV(alphas=ridge_alphas, cv=5))
])
ridge_model.fit(X_train, y_train)
ridge_preds = ridge_model.predict(X_test)
ridge_mse = mean_squared_error(y_test, ridge_preds)
ridge_alpha = ridge_model.named_steps['ridge'].alpha_
print(f'Selected lambda (alpha): {ridge_alpha:,.5f}')
print(f'Test MSE: {ridge_mse:,.2f}')
print(f'Test RMSE: {ridge_mse ** 0.5:,.2f}')


Selected lambda (alpha): 17.40209
Test MSE: 1,051,437.62
Test RMSE: 1,025.40


### Part (d)
**Original prompt:** Fit a lasso model on the training set, with lambda chosen by cross-validation. Report the test error obtained, along with the number of non-zero coefficient estimates.

In [9]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error

lasso_alphas = np.logspace(-3, 6, 400)
lasso_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', LassoCV(alphas=lasso_alphas, cv=10, max_iter=60000, random_state=42))
])
lasso_model.fit(X_train, y_train)
lasso_preds = lasso_model.predict(X_test)
lasso_mse = mean_squared_error(y_test, lasso_preds)
lasso_alpha = lasso_model.named_steps['lasso'].alpha_
non_zero = int(np.count_nonzero(lasso_model.named_steps['lasso'].coef_))
print(f'Selected lambda (alpha): {lasso_alpha:,.5f}')
print(f'Non-zero coefficients: {non_zero}')
print(f'Test MSE: {lasso_mse:,.2f}')
print(f'Test RMSE: {lasso_mse ** 0.5:,.2f}')


Selected lambda (alpha): 25.03204
Non-zero coefficients: 13
Test MSE: 1,025,034.04
Test RMSE: 1,012.44


### Part (e)
**Original prompt:** Fit a PCR model on the training set, with M chosen by cross-validation. Report the test error obtained, along with the value of M selected by cross-validation.

In [10]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

pcr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA()),
    ('regressor', LinearRegression())
])
max_components = min(16, X_train.shape[1])
param_grid = {'pca__n_components': list(range(1, max_components + 1))}
cv = KFold(n_splits=10, shuffle=True, random_state=42)
pcr_search = GridSearchCV(
    estimator=pcr_pipeline,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=cv
)
pcr_search.fit(X_train, y_train)
pcr_preds = pcr_search.predict(X_test)
pcr_mse = mean_squared_error(y_test, pcr_preds)
best_m = pcr_search.best_params_['pca__n_components']
print(f'Selected number of principal components (M): {best_m}')
print(f'Test MSE: {pcr_mse:,.2f}')
print(f'Test RMSE: {pcr_mse ** 0.5:,.2f}')


Selected number of principal components (M): 16
Test MSE: 1,071,192.41
Test RMSE: 1,034.98
